In [11]:
import pandas as pd
import numpy as np

In [12]:
data_df = pd.read_csv("typology_dataset.csv", encoding="utf-8")
data_df = data_df[data_df["political"] == "political"]
cleaned_df = data_df[["id", "account", "full_name", "biography",  "is_verified", "followers", "url", "posts", "category", "political", "typology", "typology_justification", "typology_certainty", "account_summary"]].copy()

In [13]:
print(cleaned_df.shape[0])
cleaned_df.head(1)

708


,id,account,full_name,biography,is_verified,followers,url,posts,category,political,typology,typology_justification,typology_certainty,account_summary
135,9.879575e+09,christopher_smith_nature,Christopher Smith,Information on the next steps in our fight aga...,False,12569,https://www.instagram.com/christopher_smith_na...,[{'caption': 'Thank you all so much for the su...,creative hobbies,political,left-out left,The account shows a mix of liberal views on en...,70,The account focuses on sharing wildlife photog...


In [14]:
cleaned_df["typology"].value_counts()

typology
loyal liberal                 256
leftward progressive          234
no apologies right             96
pragmatic and polite right     47
tuned-out middle               42
left-out left                  23
faith first conservative        8
unconventional right            2
Name: count, dtype: int64

In [15]:
students = [
    "sb132", "cd111", "ac163", "ct114", "os103",
    "rp104", "gb107", "ep123", "km124", "am149",
    "il115", "ec132", "ms162", "th118", "al141"
]

In [16]:
import pandas as pd

# 1. Load original cleaned dataset
df = pd.read_csv("typology_dataset.csv", encoding="utf-8")

# 2. Define student assignments per batch (6 pairs + 1 triplet = 15 students total)
student_pairs = {
    1: ("sb132", "cd111"),
    2: ("ac163", "ct114"),
    3: ("os103", "rp104"),
    4: ("gb107", "ep123"),
    5: ("km124", "am149"),
    6: ("il115", "ec132"),
    7: ("ms162", "th118", "al141"),  # Triplet batch for the 15th student
}

# 3. Exact category targets per batch (16 posts per batch)
batch_targets = {
    1: {"loyal liberal": 5, "leftward progressive": 5, "no apologies right": 2, "pragmatic and polite right": 1, "tuned-out middle": 1, "left-out left": 1, "faith first conservative": 0, "unconventional right": 1},
    2: {"loyal liberal": 5, "leftward progressive": 5, "no apologies right": 2, "pragmatic and polite right": 1, "tuned-out middle": 1, "left-out left": 1, "faith first conservative": 0, "unconventional right": 1},
    3: {"loyal liberal": 5, "leftward progressive": 5, "no apologies right": 2, "pragmatic and polite right": 1, "tuned-out middle": 1, "left-out left": 1, "faith first conservative": 1, "unconventional right": 0},
    4: {"loyal liberal": 5, "leftward progressive": 5, "no apologies right": 2, "pragmatic and polite right": 1, "tuned-out middle": 1, "left-out left": 0, "faith first conservative": 2, "unconventional right": 0},
    5: {"loyal liberal": 6, "leftward progressive": 5, "no apologies right": 2, "pragmatic and polite right": 1, "tuned-out middle": 1, "left-out left": 0, "faith first conservative": 1, "unconventional right": 0},
    6: {"loyal liberal": 6, "leftward progressive": 4, "no apologies right": 2, "pragmatic and polite right": 1, "tuned-out middle": 1, "left-out left": 0, "faith first conservative": 2, "unconventional right": 0},
    7: {"loyal liberal": 5, "leftward progressive": 5, "no apologies right": 2, "pragmatic and polite right": 1, "tuned-out middle": 0, "left-out left": 1, "faith first conservative": 2, "unconventional right": 0},
}

# 4. Stratified random sampling per category across batches 1 to 7
sampled_rows = []

for category in df["typology"].unique():
    cat_df = df[df["typology"] == category].sample(frac=1, random_state=42).reset_index(drop=True)
    idx_ptr = 0
    
    for batch_num in range(1, 8):
        count_needed = batch_targets[batch_num].get(category, 0)
        if count_needed > 0:
            batch_sample = cat_df.iloc[idx_ptr : idx_ptr + count_needed].copy()
            batch_sample["batch"] = f"Batch {batch_num}"
            sampled_rows.append(batch_sample)
            idx_ptr += count_needed

sampled_df = pd.concat(sampled_rows, ignore_index=True)

# 5. Assign to student groups dynamically (handles pairs & triplets cleanly)
final_records = []
for _, row in sampled_df.iterrows():
    batch_num = int(row["batch"].split(" ")[1])
    assigned_students = student_pairs[batch_num]
    
    for student_id in assigned_students:
        row_dict = row.to_dict()
        row_dict["user"] = student_id
        final_records.append(row_dict)

# 6. Save updated dataframe
final_df = pd.DataFrame(final_records)
final_df.to_csv("annotation_dataset.csv", index=False)